# 掘金自由流通市值字段验证

目的：验证掘金常用股票基础/市值接口里，是否存在能对齐 Tushare `daily_basic.free_share` 的字段。

核心基准口径：
- TS 自由流通市值：`free_share * close / 1e4`，单位亿元，因为 Tushare `free_share` 单位是万股。
- TS 流通市值：优先看 `ts_stock_all_data.float_mv`，单位亿元。
- GM 市值字段：掘金返回单位通常是元，换算亿元需要 `/ 1e8`。

注意：本 Notebook 只读取本地数据和调用掘金查询接口，不写入任何 parquet。

In [ ]:
%reload_ext autoreload
%autoreload 2

import datetime as dt
import pandas as pd
import polars as pl

from gm.api import (
    stk_get_daily_basic_pt,
    stk_get_daily_mktvalue_pt,
    stk_get_finance_deriv_pt,
    stk_get_finance_prime_pt,
)

from my_utils.fun import read_day_data
from my_utils.stock_api import stock_api

# 初始化项目封装的 API，主要用于设置掘金 token 和 Tushare token。
# 如果这里失败，优先检查 my_utils/stock_api.py 里的 token 是否有效。
api = stock_api()

trade_date = dt.date(2025, 11, 12)
symbols = [
    "SHSE.601288",  # 农业银行：自由流通和流通差异很大
    "SHSE.601398",  # 工商银行：大行样本
    "SHSE.600519",  # 贵州茅台：大市值白酒样本
    "SHSE.600036",  # 招商银行：银行样本
    "SHSE.601318",  # 中国平安：A/H 股样本
    "SZSE.000001",  # 平安银行：深市样本
    "SZSE.000753",  # 漳州发展：中小市值样本
    "SZSE.000858",  # 五粮液：深市大市值样本
]

print(f"trade_date = {trade_date}")
print(f"symbols = {symbols}")

## 1. 现场调用掘金基础指标接口

`stk_get_daily_basic_pt` 这里重点看股本相关字段：
- `ttl_shr`：总股本
- `circ_shr`：流通股本相关字段
- `ttl_shr_unl`：无限售总股本
- `a_shr_unl`：A 股无限售流通股本
- `h_shr_unl`：H 股无限售流通股本
- `tclose`：收盘价

如果某个 GM 股本字段能对齐 TS `free_share`，用它乘以 `tclose` 后应接近 `TS free_share * close`。

In [ ]:
gm_basic_fields = [
    "ttl_shr",
    "circ_shr",
    "ttl_shr_unl",
    "a_shr_unl",
    "h_shr_unl",
    "ttl_shr_ltd",
    "turnrate",
    "tclose",
]

gm_basic = stk_get_daily_basic_pt(
    symbols=symbols,
    fields=gm_basic_fields,
    trade_date=trade_date.strftime("%Y-%m-%d"),
    df=True,
)

gm_basic

## 2. 现场调用掘金市值接口

`stk_get_daily_mktvalue_pt` 重点看：
- `a_mv`：A 股总市值
- `a_mv_ex_ltd`：A 股流通市值，不含限售股

这两个字段返回单位是元，后续统一除以 `1e8` 转成亿元。

In [ ]:
gm_mktvalue = stk_get_daily_mktvalue_pt(
    symbols=symbols,
    fields=["a_mv", "a_mv_ex_ltd"],
    trade_date=trade_date.strftime("%Y-%m-%d"),
    df=True,
)

gm_mktvalue

## 3. 读取本地 TS 基准口径

这里不调远程 Tushare，只读取已有本地数据：
- `ts_stock_all_data.float_mv`：项目里已有的流通市值，单位亿元。
- `ts_daily_basic.free_share * close / 1e4`：原策略使用的自由流通市值，单位亿元。
- `ts_daily_basic.float_share * close / 1e4`：Tushare 流通股本推导流通市值，单位亿元。

In [ ]:
ts_stock = read_day_data(
    start_date=trade_date,
    end_date=trade_date,
    stock_list=symbols,
    file_path="ts_stock_all_data",
).select([
    "code",
    "trading_date",
    "name",
    "close",
    "float_mv",
    "total_mv",
])

ts_basic = read_day_data(
    start_date=trade_date,
    end_date=trade_date,
    stock_list=symbols,
    file_path="ts_daily_basic",
).with_columns([
    # Tushare 股本字段单位是万股；close 是元；除以 1e4 后得到亿元。
    (pl.col("free_share") * pl.col("close") / 1e4).alias("ts_free_float_mv_yi"),
    (pl.col("float_share") * pl.col("close") / 1e4).alias("ts_float_mv_from_basic_yi"),
    (pl.col("total_share") * pl.col("close") / 1e4).alias("ts_total_mv_from_basic_yi"),
]).select([
    "code",
    "trading_date",
    "free_share",
    "float_share",
    "total_share",
    "ts_free_float_mv_yi",
    "ts_float_mv_from_basic_yi",
    "ts_total_mv_from_basic_yi",
])

ts_ref = ts_stock.join(ts_basic, on=["code", "trading_date"], how="left")
ts_ref

## 4. 合并验算：GM 字段到底对齐 TS 哪个口径

判断标准：
- 如果 `gm_a_mv_ex_ltd_yi / ts_free_float_mv_yi` 接近 1，说明对齐自由流通市值。
- 如果 `gm_a_mv_ex_ltd_yi / ts_float_mv` 接近 1，说明对齐流通市值。
- 如果 `gm_a_mv_ex_ltd_yi / ts_total_mv` 接近 1，说明接近总市值。

In [ ]:
gm_basic_pl = pl.from_pandas(gm_basic).rename({"symbol": "code"}).with_columns(
    pl.col("trade_date").str.strptime(pl.Date, "%Y-%m-%d").alias("trading_date")
)

gm_mktvalue_pl = pl.from_pandas(gm_mktvalue).rename({"symbol": "code"}).with_columns(
    pl.col("trade_date").str.strptime(pl.Date, "%Y-%m-%d").alias("trading_date")
)

gm_calc = (
    gm_basic_pl
    .join(gm_mktvalue_pl, on=["code", "trading_date"], how="left")
    .with_columns([
        # GM 股本字段单位是股；价格是元；除以 1e8 后得到亿元。
        (pl.col("a_shr_unl") * pl.col("tclose") / 1e8).alias("gm_a_shr_unl_mv_yi"),
        (pl.col("circ_shr") * pl.col("tclose") / 1e8).alias("gm_circ_shr_mv_yi"),
        (pl.col("ttl_shr") * pl.col("tclose") / 1e8).alias("gm_total_shr_mv_yi"),
        (pl.col("a_mv") / 1e8).alias("gm_a_mv_yi"),
        (pl.col("a_mv_ex_ltd") / 1e8).alias("gm_a_mv_ex_ltd_yi"),
    ])
)

compare_df = (
    ts_ref
    .join(gm_calc, on=["code", "trading_date"], how="left")
    .with_columns([
        (pl.col("gm_a_mv_ex_ltd_yi") / pl.col("ts_free_float_mv_yi")).alias("gm_mktvalue_div_ts_free"),
        (pl.col("gm_a_mv_ex_ltd_yi") / pl.col("float_mv")).alias("gm_mktvalue_div_ts_float"),
        (pl.col("gm_a_shr_unl_mv_yi") / pl.col("ts_free_float_mv_yi")).alias("gm_a_shr_unl_div_ts_free"),
        (pl.col("gm_a_shr_unl_mv_yi") / pl.col("float_mv")).alias("gm_a_shr_unl_div_ts_float"),
    ])
    .select([
        "code",
        "name",
        "close",
        "ts_free_float_mv_yi",
        "float_mv",
        "total_mv",
        "gm_a_shr_unl_mv_yi",
        "gm_circ_shr_mv_yi",
        "gm_a_mv_ex_ltd_yi",
        "gm_mktvalue_div_ts_free",
        "gm_mktvalue_div_ts_float",
        "gm_a_shr_unl_div_ts_free",
        "gm_a_shr_unl_div_ts_float",
    ])
    .sort("code")
)

compare_df.to_pandas()

## 5. 自动判断最接近 TS 自由流通市值的 GM 字段

这里把几个能算出来的 GM 市值候选项，与 TS 自由流通市值计算相对误差。若某个字段是等价口径，误差应在大多数股票上接近 0。

In [ ]:
candidate_cols = [
    "gm_a_shr_unl_mv_yi",
    "gm_circ_shr_mv_yi",
    "gm_total_shr_mv_yi",
    "gm_a_mv_yi",
    "gm_a_mv_ex_ltd_yi",
]

error_rows = []
for col in candidate_cols:
    stats = compare_df.select([
        pl.lit(col).alias("gm_candidate"),
        ((pl.col(col) / pl.col("ts_free_float_mv_yi") - 1).abs()).mean().alias("mean_abs_rel_error_vs_ts_free"),
        ((pl.col(col) / pl.col("ts_free_float_mv_yi") - 1).abs()).median().alias("median_abs_rel_error_vs_ts_free"),
        ((pl.col(col) / pl.col("float_mv") - 1).abs()).mean().alias("mean_abs_rel_error_vs_ts_float_mv"),
        ((pl.col(col) / pl.col("float_mv") - 1).abs()).median().alias("median_abs_rel_error_vs_ts_float_mv"),
    ]).to_dicts()[0]
    error_rows.append(stats)

pd.DataFrame(error_rows).sort_values("median_abs_rel_error_vs_ts_free")

## 6. 候选字段名探测

这一步用于确认掘金接口是否接受类似 `free_share/free_float_share/free_float_mv` 的字段名。

如果返回 `fields 不正确`，说明当前接口不支持该字段名。

In [ ]:
def probe_fields(api_func, api_name: str, candidate_fields: list[str]) -> pd.DataFrame:
    """逐个探测字段是否被掘金接口接受，便于确认是否存在隐藏的自由流通字段。"""
    rows = []
    for field in candidate_fields:
        try:
            if api_func in [stk_get_daily_basic_pt, stk_get_daily_mktvalue_pt]:
                result = api_func(
                    symbols=["SHSE.601288"],
                    fields=[field],
                    trade_date=trade_date.strftime("%Y-%m-%d"),
                    df=True,
                )
            else:
                result = api_func(
                    symbols=["SHSE.601288"],
                    fields=field,
                    date=trade_date.strftime("%Y-%m-%d"),
                    df=True,
                )
            rows.append({
                "api": api_name,
                "field": field,
                "status": "ok",
                "shape": getattr(result, "shape", None),
                "sample": result.to_dict("records")[:1] if hasattr(result, "to_dict") else repr(result),
            })
        except Exception as exc:
            rows.append({
                "api": api_name,
                "field": field,
                "status": "error",
                "shape": None,
                "sample": str(exc)[:180],
            })
    return pd.DataFrame(rows)


share_field_candidates = [
    "free_share",
    "free_shr",
    "free_float_share",
    "free_float_shr",
    "free_circ_share",
    "free_circ_shr",
    "a_free_share",
    "a_free_shr",
    "a_free_float_share",
    "a_free_float_shr",
    "float_share",
    "float_shr",
    "circ_share",
    "circ_shr",
    "ttl_shr",
    "circ_shr",
    "ttl_shr_unl",
    "a_shr_unl",
]

mktvalue_field_candidates = [
    "free_mv",
    "free_float_mv",
    "free_circ_mv",
    "a_free_mv",
    "a_free_float_mv",
    "a_mv_free",
    "circ_mv",
    "float_mv",
    "mv_free_float",
    "mv_a_free_float",
    "a_mv",
    "a_mv_ex_ltd",
]

probe_result = pd.concat([
    probe_fields(stk_get_daily_basic_pt, "stk_get_daily_basic_pt", share_field_candidates),
    probe_fields(stk_get_daily_mktvalue_pt, "stk_get_daily_mktvalue_pt", mktvalue_field_candidates),
], ignore_index=True)

probe_result

## 7. 结论模板

运行完上面单元后，重点看：

1. `compare_df` 中农业银行、工商银行等大行的 `gm_a_mv_ex_ltd_yi` 是否接近 TS `float_mv`，而不是 `ts_free_float_mv_yi`。
2. `error_rows` 中哪个 GM 候选字段对 TS `free_share` 口径误差最小。若所有候选字段误差都很大，说明没有等价字段。
3. `probe_result` 是否存在 `free_share/free_float_*` 之类字段探测成功。

如果最终没有等价字段，策略复现 TS 口径时应继续使用 `ts_daily_basic.free_share * close / 1e4` 补充 `free_float_mv`。